# CineMatch - Week 2: Memory-Based Collaborative Filtering
This notebook implements and evaluates User-User and Item-Item collaborative filtering. It uses a deterministic per-user holdout and reports both error and prediction coverage.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from cinematch.data import load_movielens
from cinematch.memory_cf import MemoryBasedCF, cosine_similarity, deterministic_user_split, evaluate_predictions

ModuleNotFoundError: No module named 'surprise'

## 1. Cosine similarity from first principles
Cosine similarity is the dot product divided by the product of both vector magnitudes. Parallel vectors score 1, orthogonal vectors score 0, and the implementation safely returns 0 for a zero vector.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])
cosine_similarity(a, b)

## 2. Build a leakage-free evaluation split and train the model

In [ ]:
ratings, movies, _ = load_movielens(ROOT / 'data' / 'raw')
train, test = deterministic_user_split(ratings, test_fraction=0.2, seed=42)
model = MemoryBasedCF(k_neighbors=30, min_common=3).fit(train)
print(f'Train: {len(train):,} | Held out: {len(test):,}')

## 3. Inspect similar users and movies

In [ ]:
display(model.similar_users(user_id=1, n=10))
display(model.similar_items(movie_id=260, movies=movies, n=10))

## 4. Evaluate both methods on the identical held-out sample
The command-line run saves verified metrics to `data/processed/week2_summary.json`. Loading that artifact keeps the notebook aligned with the tested run.

In [ ]:
summary = json.loads((ROOT / 'data' / 'processed' / 'week2_summary.json').read_text())
pd.DataFrame({'User-User': summary['user_user'], 'Item-Item': summary['item_item']})

## 5. Tested recommendations
The demo model is trained on the full history so an already-rated movie cannot be presented as unseen. Candidate generation is limited to the 500 most-rated unseen movies for stable collaborative-filtering coverage.

In [ ]:
display(pd.read_csv(ROOT / 'data' / 'processed' / 'week2_user_user_recommendations_user_1.csv'))
display(pd.read_csv(ROOT / 'data' / 'processed' / 'week2_item_item_recommendations_user_1.csv'))